In [3]:
import pandas as pd
from shapley_attribution import (
    MonteCarloShapleyAttribution,
    LinearAttribution,
    make_attribution_problem,
    attribution_summary,
)

# Generate synthetic data with known ground truth
journeys, conversions, ground_truth, channels = make_attribution_problem(
    n_channels=8, n_journeys=5000, random_state=42
)

print(f"Journeys: {len(journeys)}, Conversions: {conversions.sum()}/{len(conversions)} ({conversions.mean():.0%})")
print(f"Ground truth: {ground_truth.round(4)}")

# Fit a model (scikit-learn style) — pass conversion labels
model = MonteCarloShapleyAttribution(n_iter=2000, random_state=42)
model.fit(journeys, y=conversions)

# Get aggregate attribution scores
scores = model.get_attribution()        # dict: channel -> score
array  = model.get_attribution_array()  # numpy array aligned with model.channels_

# Per-journey attribution matrix
matrix = model.transform(journeys)      # shape (n_journeys, n_channels)

# Compare multiple models
linear = LinearAttribution().fit(journeys, y=conversions)
results = pd.DataFrame(attribution_summary(
    {"MC Shapley": model.get_attribution_array(),
     "Linear": linear.get_attribution_array()},
    ground_truth
))

results

Journeys: 5000, Conversions: 1510/5000 (30%)
Ground truth: [0.132  0.0716 0.0647 0.0647 0.3129 0.1669 0.0498 0.1375]


,MC Shapley,Linear
nmae,0.022057,0.045654
rank_correlation,0.976190,0.952381
top_3_overlap,1.000000,1.000000
